# Shack–Hartmann: from pupil to closed loop

This tutorial validates the complete deterministic chain: modal OPD, lenslet spots, calibrated slopes, interaction matrix, SVD reconstruction and closed-loop correction. OPD commands are in metres and slopes are in radians.

In [ ]:
import torch
import matplotlib.pyplot as plt

from fiatlux import (
    ActuatorGrid, DeformableMirror, Grid, InteractionMatrix, PlaneWave,
    ShackHartmannLensletArray, ShackHartmannSlopeEstimator, Spectrum,
    ZernikeBasis,
)
from fiatlux.core.spectrum import Band

## 1. Optical bench

A 24×24 pupil is divided into 6×6 subapertures. Two mirrors use the same tip, tilt and defocus basis: one injects the unknown OPD and one corrects it.

In [ ]:
dtype = torch.float64
grid = Grid(24, 24, 0.05, 0.05, dtype=dtype)
spectrum = Spectrum(
    magnitude=0, band=Band(650e-9, 0.0, 368.0), samples=1, dtype=dtype
)
source_field = PlaneWave(spectrum).generate_field(grid)
actuators = ActuatorGrid(1, 1, 1.0)

def make_mirror():
    return DeformableMirror(
        grid, actuators, grid, ZernikeBasis(grid, n=3), stroke=500e-9
    )

aberration = make_mirror()
correction = make_mirror()
lenslets = ShackHartmannLensletArray(
    grid, pitch=0.20, focal_length=2.0, spot_oversampling=4
)
estimator = ShackHartmannSlopeEstimator(focal_length=2.0)

def acquire(field, mirror=correction):
    return estimator.measure(lenslets.propagate(mirror.apply(field))).slope_vector

flat_spots = lenslets.propagate(source_field)
flat_slopes = estimator.measure(flat_spots).slope_vector
assert torch.allclose(flat_slopes, torch.zeros_like(flat_slopes))
print(f"Measurement vector: {flat_slopes.shape[0]} slopes")

## 2. Flat and aberrated spot mosaics

In [ ]:
injected = torch.tensor([30e-9, -20e-9, 15e-9], dtype=dtype)
aberration.commands = injected
aberrated_field = aberration.apply(source_field)
aberrated_spots = lenslets.propagate(aberrated_field)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, image, title in zip(
    axes,
    [flat_spots.mosaic(), aberrated_spots.mosaic()],
    ["Flat wavefront", "Tip + tilt + defocus"],
):
    view = image.detach().cpu()
    ax.imshow(view / view.max(), origin="lower", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("detector x pixel")
    ax.set_ylabel("detector y pixel")
fig.tight_layout()

## 3. Push–pull calibration and reconstruction

Each correction mode is poked by ±10 nm OPD. The control matrix is a filtered SVD pseudo-inverse.

In [ ]:
calibration = InteractionMatrix(
    correction,
    lambda: acquire(source_field),
    poke_amplitude=10e-9,
)
interaction = calibration.calibrate_push_pull(verbose=False)
control = calibration.compute_control_matrix(rcond=1e-5)
print("Interaction matrix:", tuple(interaction.shape))
print("Retained rank:", calibration.effective_rank)

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(calibration.singular_values.detach().cpu(), "o-")
ax.set(xlabel="mode", ylabel="singular value", title="SH interaction matrix")
ax.grid(True)

In [ ]:
measured = acquire(aberrated_field)
reconstructed = control @ measured
print("Injected [nm]:     ", injected.mul(1e9).tolist())
print("Reconstructed [nm]:", reconstructed.mul(1e9).tolist())
torch.testing.assert_close(reconstructed, injected, rtol=0.04, atol=0.2e-9)

## 4. Closed loop

The reconstructor sees only measured slopes. With an integrator gain of 0.7, the correction mirror converges toward the negative injected OPD.

In [ ]:
correction.flatten()
residuals = []
for iteration in range(7):
    residual_field = correction.apply(aberration.apply(source_field))
    measurement = estimator.measure(lenslets.propagate(residual_field))
    residuals.append(float(torch.linalg.vector_norm(measurement.slope_vector)))
    correction.commands = correction.commands - 0.7 * (control @ measurement.slope_vector)

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(residuals, "o-")
ax.set(xlabel="iteration", ylabel="slope-vector norm [rad]", title="Closed-loop convergence")
ax.grid(True)
print(f"Residual reduction: {residuals[0] / residuals[-1]:.1f}×")
assert residuals[-1] < 0.02 * residuals[0]

The example is intentionally noise-free so the analytical response and convergence can be checked quantitatively. A noisy exposure can be inserted between the lenslet array and estimator with ShackHartmannDetector, as described in the Shack–Hartmann documentation.